# ANM-OpenFold3: Single Protein Run
Tek protein icin mode-drive pipeline calistirma notebook'u.

**Giris parametreleri:**
- `INITIAL_PDB` / `TARGET_PDB` — baslangic ve hedef PDB ID'leri
- `CHAIN` — chain ID (multimer icin `"A,B"`)
- `UNIPROT_ID` — (opsiyonel) UniProt ID, verilirse 3. mod olarak calisir
- `RUN_NAME` — sonuclarin kaydedilecegi isim

Monomer ve multimer destekler. V2 config + selective mixing.

In [ ]:
# ════════════════════════════════════════════════════════════════
#  CELL 1: Environment Setup
# ════════════════════════════════════════════════════════════════

import os, sys, shutil, time, json, warnings
warnings.filterwarnings('ignore')

IN_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)

    if os.path.exists('/content/ANM-openfold3'):
        shutil.rmtree('/content/ANM-openfold3')
    !git clone https://github.com/beratkaskaloglu/ANM-openfold3.git /content/ANM-openfold3

    if not os.path.exists('/content/ANM-openfold3/openfold3-repo'):
        !git clone https://github.com/aqlaboratory/openfold-3.git /content/ANM-openfold3/openfold3-repo
        !pip install -e /content/ANM-openfold3/openfold3-repo -q
    else:
        try:
            import openfold3
        except ImportError:
            !pip install -e /content/ANM-openfold3/openfold3-repo -q

    !pip install -q biopython matplotlib numpy torch pandas requests

    _repo_root = '/content/ANM-openfold3'
else:
    _repo_root = os.path.abspath(os.path.join(os.getcwd(), '..'))

if _repo_root not in sys.path:
    sys.path.insert(0, _repo_root)

if IN_COLAB:
    of3_root = '/content/ANM-openfold3/openfold3-repo'
    if of3_root not in sys.path:
        sys.path.insert(0, of3_root)
    os.makedirs(os.path.expanduser('~/.openfold3'), exist_ok=True)
    from openfold3.entry_points.parameters import (
        download_model_parameters, get_default_checkpoint_dir, DEFAULT_CHECKPOINT_NAME,
    )
    _param_dir = get_default_checkpoint_dir()
    _param_dir.mkdir(parents=True, exist_ok=True)
    download_model_parameters(_param_dir, DEFAULT_CHECKPOINT_NAME, skip_confirmation=True)

print(f'Repo root: {_repo_root}')
print('Environment ready.')

## 2. Kullanici Girdileri

In [ ]:
# ════════════════════════════════════════════════════════════════
#  KULLANICI GIRDILERI — Buraya kendi proteinini yaz
# ════════════════════════════════════════════════════════════════

# PDB yapilari
INITIAL_PDB = '4AKE'        # Baslangic yapisi (open)
TARGET_PDB  = '1AKE'        # Hedef yapisi (closed)
CHAIN       = 'A'           # Chain ID (multimer icin: 'A,B')

# UniProt (opsiyonel — bos birak = sadece PDB sekans ile calisir)
UNIPROT_ID  = 'P69441'      # '' birakirsan UniProt modu skip edilir

# Sonuc kayit ismi
RUN_NAME    = 'adenylate_kinase'

print(f'Initial: {INITIAL_PDB} (chain {CHAIN})')
print(f'Target:  {TARGET_PDB}')
print(f'UniProt: {UNIPROT_ID or "(yok)"}')
print(f'Run:     {RUN_NAME}')

## 3. Pipeline Konfigurasyonu

In [ ]:
# ════════════════════════════════════════════════════════════════
#  CELL 3: Pipeline Configuration
# ════════════════════════════════════════════════════════════════

N_STEPS = 20

# Selective mixing (V6 best)
SELECTIVE_MIXING     = True
CHANGE_CUTOFF        = 0.15
ALPHA_BASE           = 0.05
ALPHA_MAX            = 0.7
MAPPING              = 'sigmoid'
W_CONNECTIVITY       = 0.5
W_DISTANCE           = 0.5
DIAGONAL_BAND        = 2

# Pipeline
Z_MIXING_ALPHA       = 0.30
COMBINATION_STRATEGY = 'autostop'
N_ANM_MODES          = 20
N_COMBINATIONS       = 20
MAX_COMBO_SIZE       = 3
DF                   = 0.6
DF_MIN               = 0.3
DF_MAX               = 3.0
ANM_CUTOFF           = 15.0
CONTACT_R_CUT        = 10.0
CONTACT_TAU          = 1.5
Z_DIRECTION          = 'plus'

# Composite gate
USE_COMPOSITE_GATE       = True
COMPOSITE_GATE_THRESHOLD = 0.55
GATE_W_CR                = 0.45
GATE_W_PLDDT             = 0.30
GATE_W_RG                = 0.15
GATE_W_PTM               = 0.10

# Hard cutoffs
CONFIDENCE_PTM_CUTOFF     = 0.15
CONFIDENCE_PLDDT_CUTOFF   = 65.0
CONFIDENCE_RG_MAX         = 2.5
CONFIDENCE_RG_MIN         = 0.3
CONFIDENCE_CLASH_REJECT   = True
CONFIDENCE_RMSD_INIT_MAX  = 10.0

# Stall & drift
ALPHA_DECAY           = 0.8
MAX_STALL             = 8
ENABLE_BEST_ROLLBACK  = True
BEST_ROLLBACK_TM_DROP = 0.40
ENABLE_ADAPTIVE_STOP  = True
ADAPTIVE_STOP_WINDOW  = 3
MAX_ROLLBACK_STOP     = 5

# Fallback
ENABLE_FALLBACK           = True
FALLBACK_EXTENDED_ENABLED = True
AUTOSTOP_FALLBACK_LEVELS  = (0, 1, 4, 9)

# Autostop
AUTOSTOP_V_MAGNITUDE = 1.0
AUTOSTOP_N_STEPS     = 5000
AUTOSTOP_BACK_OFF    = 2

# OF3
USE_MSA_SERVER        = True
USE_TEMPLATES         = False
NUM_ROLLOUT_STEPS     = 5
NUM_DIFFUSION_SAMPLES = 1

# Converter
DRIVE_DIR = '/content/drive/MyDrive/ANM-openfold3/checkpoints/finetune_msa'
CHECKPOINT_SELECTION = 'best_bf_r'

# Output
if IN_COLAB:
    BASE_SAVE_DIR = '/content/drive/MyDrive/ANM-openfold3/benchmark_v2'
else:
    BASE_SAVE_DIR = os.path.join(_repo_root, 'results', 'benchmark_v2')
os.makedirs(BASE_SAVE_DIR, exist_ok=True)

# Skip mode C (UniProt) for multi-chain entries
SKIP_UNIPROT_MULTIMER = True

print(f'N_STEPS={N_STEPS}, Z_MIXING_ALPHA={Z_MIXING_ALPHA}, selective={SELECTIVE_MIXING}')
print(f'NUM_ROLLOUT_STEPS={NUM_ROLLOUT_STEPS}, NUM_DIFFUSION_SAMPLES={NUM_DIFFUSION_SAMPLES}')
print(f'SKIP_UNIPROT_MULTIMER={SKIP_UNIPROT_MULTIMER}')
print(f'Save dir: {BASE_SAVE_DIR}')

## 4. Helper Fonksiyonlar

In [ ]:
# ════════════════════════════════════════════════════════════════
#  CELL 4: Helper Functions
# ════════════════════════════════════════════════════════════════

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import requests
matplotlib.rcParams['figure.dpi'] = 120

from Bio.PDB import PDBParser, PDBList
from Bio.SeqUtils import seq1

# Reload src modules to pick up latest changes
for mod_name in list(sys.modules.keys()):
    if mod_name.startswith('src'):
        del sys.modules[mod_name]

from src.converter import PairContactConverter
from src.mode_drive import ModeDrivePipeline, ModeDriveConfig, compute_rmsd, tm_score
from src.mode_drive_utils import align_and_trim_ca
from src.of3_diffusion import load_of3_model, OF3ModelCache
from src.autostop_adapter import StructureContext


# ────────────────────────────────────────────────────────────────
#  PDB Fetch Helpers
# ────────────────────────────────────────────────────────────────

def fetch_ca(pdb_id: str, chain_id: str):
    """PDB'den CA koordinatlarini ve sekansini cek. Chain fallback var."""
    parser = PDBParser(QUIET=True)
    pdbl = PDBList()
    pdb_file = pdbl.retrieve_pdb_file(pdb_id, pdir='/tmp/pdb_cache', file_format='pdb')
    structure = parser.get_structure(pdb_id, pdb_file)

    chains = [c for c in structure[0].get_chains() if c.id == chain_id]
    if not chains:
        available = [c.id for c in structure[0].get_chains()]
        if available:
            print(f'    WARNING: chain {chain_id} not found in {pdb_id}, using {available[0]} (available: {available})')
            chain = [c for c in structure[0].get_chains() if c.id == available[0]][0]
        else:
            raise ValueError(f'No chains in {pdb_id}')
    else:
        chain = chains[0]

    residues = [r for r in chain if r.get_id()[0] == ' ' and 'CA' in r]
    coords = torch.tensor([r['CA'].get_vector().get_array() for r in residues], dtype=torch.float32)
    sequence = ''.join(seq1(r.get_resname()) for r in residues)
    return coords, sequence


def fetch_ca_multimer(pdb_id, chain_ids_str):
    """Fetch CA coords for one or more chains, concatenated.

    Args:
        pdb_id: PDB ID
        chain_ids_str: "A" or "A,B" comma-separated

    Returns:
        coords: [N_total, 3] tensor
        sequence: concatenated sequence string
        chain_breaks: list of ints — start index of each chain segment [0, N_A, N_A+N_B]
        chain_seqs: list of per-chain sequences
    """
    chain_ids = [c.strip() for c in chain_ids_str.split(",")]
    all_coords = []
    all_seqs = []
    chain_breaks = [0]

    for cid in chain_ids:
        ca, seq = fetch_ca(pdb_id, cid)
        all_coords.append(ca)
        all_seqs.append(seq)
        chain_breaks.append(chain_breaks[-1] + len(seq))

    coords = torch.cat(all_coords, dim=0)
    sequence = "".join(all_seqs)
    return coords, sequence, chain_breaks, all_seqs


def fetch_uniprot_sequence(uniprot_id: str) -> str:
    """UniProt'tan full protein sekansini cek."""
    url = f'https://rest.uniprot.org/uniprotkb/{uniprot_id}.fasta'
    resp = requests.get(url, timeout=30)
    resp.raise_for_status()
    lines = resp.text.strip().split('\n')
    seq = ''.join(l.strip() for l in lines if not l.startswith('>'))
    print(f'    UniProt {uniprot_id}: {len(seq)} residues')
    return seq

def fetch_pdb_ids_for_uniprot(uniprot_id, max_results=50):
    """RCSB Search API ile UniProt ID'ye ait tum PDB ID'lerini cek."""
    query = {
        "query": {
            "type": "terminal",
            "service": "text",
            "parameters": {
                "attribute": "rcsb_polymer_entity_container_identifiers.reference_sequence_identifiers.database_accession",
                "operator": "exact_match",
                "value": uniprot_id
            }
        },
        "request_options": {
            "return_all_hits": True,
            "results_content_type": ["experimental"]
        },
        "return_type": "entry"
    }
    resp = requests.post(
        'https://search.rcsb.org/rcsbsearch/v2/query',
        json=query, timeout=30
    )
    if resp.status_code != 200:
        return []
    data = resp.json()
    pdb_ids = [hit['identifier'] for hit in data.get('result_set', [])]
    return pdb_ids[:max_results]




# ────────────────────────────────────────────────────────────────
#  UniProt Diffusion Wrapper
# ────────────────────────────────────────────────────────────────

def make_uniprot_diffusion_wrapper(diffusion_fn, zij_trunk_full, pdb_indices, n_pdb):
    """UniProt modda diffusion_fn'i wrap et.

    Pipeline z_mod [N_pdb, N_pdb, C] uretir. Bunu full UniProt boyutuna
    genisletip diffusion_fn'e ver, sonra donen CA'yi PDB pozisyonlarina indir.
    """
    idx_t = torch.tensor(pdb_indices, dtype=torch.long)
    n_uni = zij_trunk_full.shape[0]

    def _wrapped_diffusion(z_mod_pdb):
        z_full = zij_trunk_full.clone()
        z_full[idx_t.unsqueeze(1), idx_t.unsqueeze(0), :] = z_mod_pdb
        result = diffusion_fn(z_full)
        if hasattr(result, 'best_ca'):
            result.best_ca = result.best_ca[idx_t]
            result.all_ca = result.all_ca[:, idx_t, :]
            if result.plddt is not None:
                result.plddt = result.plddt[:, idx_t] if result.plddt.dim() == 2 else result.plddt[idx_t]
        else:
            result = result[idx_t]
        return result

    return _wrapped_diffusion


# ────────────────────────────────────────────────────────────────
#  Converter / Config Loaders
# ────────────────────────────────────────────────────────────────

def load_converter():
    """Converter checkpoint yukle."""
    history_path = os.path.join(DRIVE_DIR, 'history.json')
    best_model_path = os.path.join(DRIVE_DIR, 'best_model.pt')

    if CHECKPOINT_SELECTION == 'best_bf_r' and os.path.exists(history_path):
        with open(history_path) as f:
            history = json.load(f)
        best_bf_r = -1
        best_epoch = -1
        for entry in history:
            bf_r = entry.get('val_bf_pearson', 0)
            if bf_r > best_bf_r:
                best_bf_r = bf_r
                best_epoch = entry['epoch']
        epoch_ckpt = os.path.join(DRIVE_DIR, f'epoch_{best_epoch:04d}.pt')
        ckpt_path = epoch_ckpt if os.path.exists(epoch_ckpt) else best_model_path
        print(f'  Converter: best bf_r={best_bf_r:.4f} epoch={best_epoch}')
    else:
        ckpt_path = best_model_path
    return PairContactConverter(ckpt_path)


def make_config(chain_id):
    """Pipeline config olustur."""
    return ModeDriveConfig(
        n_steps=N_STEPS,
        combination_strategy=COMBINATION_STRATEGY,
        z_mixing_alpha=Z_MIXING_ALPHA,
        n_anm_modes=N_ANM_MODES,
        n_combinations=N_COMBINATIONS,
        max_combo_size=MAX_COMBO_SIZE,
        df=DF, df_min=DF_MIN, df_max=DF_MAX,
        anm_cutoff=ANM_CUTOFF,
        contact_r_cut=CONTACT_R_CUT,
        contact_tau=CONTACT_TAU,
        z_direction=Z_DIRECTION,
        num_diffusion_samples=NUM_DIFFUSION_SAMPLES,
        use_composite_gate=USE_COMPOSITE_GATE,
        composite_gate_threshold=COMPOSITE_GATE_THRESHOLD,
        gate_w_cr=GATE_W_CR,
        gate_w_plddt=GATE_W_PLDDT,
        gate_w_rg=GATE_W_RG,
        gate_w_ptm=GATE_W_PTM,
        confidence_ptm_cutoff=CONFIDENCE_PTM_CUTOFF,
        confidence_plddt_cutoff=CONFIDENCE_PLDDT_CUTOFF,
        confidence_rg_max=CONFIDENCE_RG_MAX,
        confidence_rg_min=CONFIDENCE_RG_MIN,
        confidence_clash_reject=CONFIDENCE_CLASH_REJECT,
        confidence_rmsd_init_max=CONFIDENCE_RMSD_INIT_MAX,
        enable_confidence_fallback=ENABLE_FALLBACK,
        fallback_extended_enabled=FALLBACK_EXTENDED_ENABLED,
        autostop_v_magnitude=AUTOSTOP_V_MAGNITUDE,
        autostop_n_steps=AUTOSTOP_N_STEPS,
        autostop_back_off=AUTOSTOP_BACK_OFF,
        autostop_fallback_levels=AUTOSTOP_FALLBACK_LEVELS,
        autostop_chain_id=chain_id,
        rejected_alpha_decay=ALPHA_DECAY,
        max_consecutive_rejected=MAX_STALL,
        confidence_warmup_steps=0,
        enable_best_rollback=ENABLE_BEST_ROLLBACK,
        best_rollback_tm_drop=BEST_ROLLBACK_TM_DROP,
        enable_adaptive_stop=ENABLE_ADAPTIVE_STOP,
        adaptive_stop_window=ADAPTIVE_STOP_WINDOW,
        selective_mixing=SELECTIVE_MIXING,
        selective_w_connectivity=W_CONNECTIVITY,
        selective_w_distance=W_DISTANCE,
        selective_change_cutoff=CHANGE_CUTOFF,
        selective_alpha_base=ALPHA_BASE,
        selective_alpha_max=ALPHA_MAX,
        selective_mapping=MAPPING,
        selective_diagonal_band=DIAGONAL_BAND,
    )


# ────────────────────────────────────────────────────────────────
#  Main Pipeline Runner
# ────────────────────────────────────────────────────────────────

def run_single_direction(converter, of3_cache, pair, direction='s1_to_s2',
                         mode='directed', uniprot_seq=None, save_dir_override=None):
    """Run pipeline for one direction.

    Args:
        pair: dict from BENCHMARK_PAIRS
        direction: 's1_to_s2' or 's2_to_s1'
        mode: 'directed' or 'uniprot'
        uniprot_seq: full UniProt sequence (for mode='uniprot')

    Returns:
        result dict with TM/RMSD metrics, or None on failure
    """
    if direction == 's1_to_s2':
        init_pdb, init_chain = pair["pdb_s1"], pair["chain_s1"]
        tgt_pdb, tgt_chain = pair["pdb_s2"], pair["chain_s2"]
        label = f'{pair["label_s1"]} -> {pair["label_s2"]}'
    else:
        init_pdb, init_chain = pair["pdb_s2"], pair["chain_s2"]
        tgt_pdb, tgt_chain = pair["pdb_s1"], pair["chain_s1"]
        label = f'{pair["label_s2"]} -> {pair["label_s1"]}'

    is_multichain = "," in init_chain

    save_dir = save_dir_override or os.path.join(BASE_SAVE_DIR, f'{pair["idx"]:02d}_{pair["name"][:20].replace(" ", "_")}')
    os.makedirs(save_dir, exist_ok=True)
    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    # ── Fetch structures ──
    if is_multichain:
        initial_ca, seq_initial, chain_breaks_init, chain_seqs_init = fetch_ca_multimer(init_pdb, init_chain)
        target_ca_raw, seq_target, chain_breaks_tgt, chain_seqs_tgt = fetch_ca_multimer(tgt_pdb, tgt_chain)
    else:
        initial_ca, seq_initial = fetch_ca(init_pdb, init_chain)
        target_ca_raw, seq_target = fetch_ca(tgt_pdb, tgt_chain)
        chain_seqs_init = [seq_initial]

    N_init = initial_ca.shape[0]
    N_target = target_ca_raw.shape[0]

    # ── Size mismatch handling ──
    if N_init != N_target:
        print(f'    SIZE MISMATCH: initial N={N_init}, target N={N_target} -> alignment')
        ca_init_trim, ca_tgt_trim, common_seq, idx_init, idx_target = align_and_trim_ca(
            initial_ca, seq_initial, target_ca_raw, seq_target
        )
        print(f'    Common core: {len(common_seq)} residues')
        target_ca = ca_tgt_trim
        initial_ca_for_comparison = ca_init_trim
        comparison_indices = idx_init
    else:
        target_ca = target_ca_raw
        initial_ca_for_comparison = initial_ca
        comparison_indices = None

    init_tm = tm_score(initial_ca_for_comparison, target_ca)
    init_rmsd = compute_rmsd(initial_ca_for_comparison, target_ca)
    print(f'    N={N_init}, init TM={init_tm:.4f}, init RMSD={init_rmsd:.2f}A ({label})')

    # ── OF3 sequence selection & diffusion_fn ──
    if mode == 'uniprot' and uniprot_seq is not None:
        of3_sequence = uniprot_seq
        N_of3 = len(of3_sequence)
        print(f'    OF3 sequence: UniProt ({N_of3} res) vs PDB ({N_init} res)')

        diffusion_fn_full, zij_trunk_full = of3_cache.prepare_sequence(
            sequence=of3_sequence,
            query_name=init_pdb,
            chain_id=init_chain.split(",")[0] if is_multichain else init_chain,
            num_rollout_steps=NUM_ROLLOUT_STEPS,
        )

        if N_of3 != N_init:
            from Bio.Align import PairwiseAligner
            _aligner = PairwiseAligner()
            _aligner.mode = 'global'
            _aligner.match_score = 2
            _aligner.mismatch_score = -1
            _aligner.open_gap_score = -5
            _aligner.extend_gap_score = -0.5
            _aln = _aligner.align(of3_sequence, seq_initial)[0]

            pdb_in_uniprot = []
            pos_uni, pos_pdb = 0, 0
            for char_u, char_p in zip(str(_aln[0]), str(_aln[1])):
                gap_u = (char_u == '-')
                gap_p = (char_p == '-')
                if not gap_u and not gap_p:
                    pdb_in_uniprot.append(pos_uni)
                if not gap_u:
                    pos_uni += 1
                if not gap_p:
                    pos_pdb += 1

            print(f'    PDB->UniProt mapping: {len(pdb_in_uniprot)} positions matched')

            idx_t = torch.tensor(pdb_in_uniprot, dtype=torch.long)
            zij_trunk = zij_trunk_full[idx_t][:, idx_t, :]
            print(f'    zij_trunk trimmed: {zij_trunk.shape}')

            diffusion_fn = make_uniprot_diffusion_wrapper(
                diffusion_fn_full, zij_trunk_full, pdb_in_uniprot, N_init
            )
        else:
            zij_trunk = zij_trunk_full
            diffusion_fn = diffusion_fn_full
    elif is_multichain:
        # Multi-chain OF3 query
        chains_data = [
            {"chain_id": cid.strip(), "sequence": seq}
            for cid, seq in zip(init_chain.split(","), chain_seqs_init)
        ]
        diffusion_fn, zij_trunk = of3_cache.prepare_sequence(
            sequence=seq_initial,  # ignored when chains_data is set
            query_name=init_pdb,
            chain_id=init_chain.split(",")[0],
            num_rollout_steps=NUM_ROLLOUT_STEPS,
            chains_data=chains_data,
        )
    else:
        # Single-chain directed mode
        diffusion_fn, zij_trunk = of3_cache.prepare_sequence(
            sequence=seq_initial,
            query_name=init_pdb,
            chain_id=init_chain,
            num_rollout_steps=NUM_ROLLOUT_STEPS,
        )

    # ── Structure context (for autostop) ──
    _pdb_file = PDBList().retrieve_pdb_file(init_pdb, pdir='/tmp/pdb_cache', file_format='pdb')
    _autostop_chain = init_chain.split(",")[0] if is_multichain else init_chain
    structure_ctx = StructureContext.from_pdb(_pdb_file, chain_id=_autostop_chain)

    # ── Config & Pipeline ──
    cfg = make_config(_autostop_chain)
    pipe = ModeDrivePipeline(
        converter=converter, config=cfg,
        diffusion_fn=diffusion_fn,
        structure_ctx=structure_ctx,
    )

    # ── Step loop ──
    coords_ca = initial_ca.clone()
    z_current = zij_trunk.clone()
    orig_alpha = Z_MIXING_ALPHA
    current_alpha = orig_alpha
    consecutive_rejected = 0
    coords_best = initial_ca.clone()
    z_best = zij_trunk.clone()
    best_target_tm = init_tm
    rollback_count = 0

    trajectory_coords = [initial_ca.clone()]
    step_tms_target = [init_tm]
    step_rmsds_target = [init_rmsd]
    step_tms_init = []
    step_rmsds_init = []
    step_tms_source = [1.0]
    step_rmsds_source = [0.0]
    first_predicted_ca = None
    step_metrics = []

    t0 = time.time()

    for step_idx in range(N_STEPS):
        cfg.z_mixing_alpha = current_alpha
        sr = pipe.step_with_fallback(
            coords_ca, initial_ca, z_current,
            prev_rmsd=0.0,
            target_coords=target_ca if comparison_indices is None else None,
            step_idx=step_idx,
        )

        accepted = not sr.rejected
        if sr.rg_ratio is not None and sr.rg_ratio > CONFIDENCE_RG_MAX:
            accepted = False
        if sr.has_clash and CONFIDENCE_CLASH_REJECT:
            accepted = False
        if sr.rmsd > CONFIDENCE_RMSD_INIT_MAX:
            accepted = False

        # Trim predicted CA for comparison
        predicted_ca_cmp = sr.new_ca[comparison_indices] if comparison_indices is not None else sr.new_ca

        cur_tm = tm_score(predicted_ca_cmp, target_ca)
        cur_rmsd = compute_rmsd(predicted_ca_cmp, target_ca)
        cur_tm_source = tm_score(predicted_ca_cmp, initial_ca_for_comparison)
        cur_rmsd_source = compute_rmsd(predicted_ca_cmp, initial_ca_for_comparison)

        # First predict baseline
        if first_predicted_ca is None and accepted:
            first_predicted_ca = predicted_ca_cmp.clone()

        if first_predicted_ca is not None:
            tm_vs_first = tm_score(predicted_ca_cmp, first_predicted_ca)
            rmsd_vs_first = compute_rmsd(predicted_ca_cmp, first_predicted_ca)
        else:
            tm_vs_first = 1.0
            rmsd_vs_first = 0.0

        plddt_mean = float(sr.plddt.mean()) if sr.plddt is not None else None

        m = {
            'step': step_idx + 1,
            'accepted': accepted,
            'tm_to_target': cur_tm,
            'rmsd_to_target': cur_rmsd,
            'tm_to_first': tm_vs_first,
            'rmsd_to_first': rmsd_vs_first,
            'rmsd_to_init': sr.rmsd,
            'ptm': sr.ptm,
            'plddt_mean': plddt_mean,
            'contact_recon': sr.contact_recon,
            'rg_ratio': sr.rg_ratio,
            'fallback_level': sr.fallback_level,
            'alpha_used': current_alpha,
        }
        step_metrics.append(m)

        tag = 'ACC' if accepted else 'REJ'
        print(
            f'      [{step_idx+1:>2}/{N_STEPS}] {tag} '
            f'TM_tgt={cur_tm:.3f} RMSD_tgt={cur_rmsd:.1f} '
            f'TM_1st={tm_vs_first:.3f} FL={sr.fallback_level}'
        )

        if accepted:
            trajectory_coords.append(sr.new_ca.clone())
            step_tms_target.append(cur_tm)
            step_rmsds_target.append(cur_rmsd)
            step_tms_init.append(tm_vs_first)
            step_rmsds_init.append(rmsd_vs_first)
            step_tms_source.append(cur_tm_source)
            step_rmsds_source.append(cur_rmsd_source)
            coords_ca = sr.new_ca
            z_current = sr.z_modified
            consecutive_rejected = 0
            current_alpha = orig_alpha

            # Best tracking
            if cur_tm > best_target_tm:
                best_target_tm = cur_tm
                coords_best = sr.new_ca.clone()
                z_best = sr.z_modified.clone()
            elif best_target_tm > 0 and cur_tm < best_target_tm * (1.0 - BEST_ROLLBACK_TM_DROP):
                coords_ca = coords_best.clone()
                z_current = z_best.clone()
                rollback_count += 1
                if rollback_count >= MAX_ROLLBACK_STOP:
                    print(f'      STOP: {rollback_count} rollback')
                    break
        else:
            consecutive_rejected += 1
            if ALPHA_DECAY < 1.0:
                current_alpha = max(0.02, current_alpha * ALPHA_DECAY)
            if MAX_STALL > 0 and consecutive_rejected >= MAX_STALL:
                print(f'      STALL: {consecutive_rejected} rejected')
                break

    wall = time.time() - t0

    # ── Result ──
    final_tm = step_tms_target[-1]
    final_rmsd = step_rmsds_target[-1]
    n_accepted = len(trajectory_coords) - 1

    result = {
        'init_tm': init_tm,
        'init_rmsd': init_rmsd,
        'best_tm': best_target_tm,
        'final_tm': final_tm,
        'final_rmsd': final_rmsd,
        'delta_tm': best_target_tm - init_tm,
        'n_accepted': n_accepted,
        'n_steps_run': len(step_metrics),
        'rollbacks': rollback_count,
        'wall_time': wall,
        'step_tms_target': step_tms_target,
        'step_rmsds_target': step_rmsds_target,
        'step_tms_init': step_tms_init,
        'step_rmsds_init': step_rmsds_init,
        'step_tms_source': step_tms_source,
        'step_rmsds_source': step_rmsds_source,
        'step_metrics': step_metrics,
        'trajectory_coords': trajectory_coords,
        'mode': mode,
    }

    print(f'    => init_TM={init_tm:.4f} best_TM={best_target_tm:.4f} delta={best_target_tm-init_tm:+.4f} ({wall:.0f}s)')
    return result


print('Helper fonksiyonlar tanimlandi.')

## 5. Model Yukle

In [ ]:
# ════════════════════════════════════════════════════════════════
#  CELL 5: Load Converter + OF3 Model (bir kez)
# ════════════════════════════════════════════════════════════════

converter = load_converter()
print('Converter ready.')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
of3_cache = load_of3_model(
    device=device,
    num_samples=NUM_DIFFUSION_SAMPLES,
    use_msa_server=USE_MSA_SERVER,
    use_templates=USE_TEMPLATES,
)
print('OF3 model cached — her protein icin prepare_sequence() kullanilacak.')

## 6. Pipeline Calistir

In [ ]:
# ════════════════════════════════════════════════════════════════
#  PIPELINE CALISTIR: 2 veya 3 mod
#  [A] Initial -> Target (PDB sekans)
#  [B] Target -> Initial (PDB sekans)
#  [C] UniProt sekans (opsiyonel)
# ════════════════════════════════════════════════════════════════

import pickle

# Pair dict olustur (V2 formatinda)
pair = {
    'idx': 1,
    'name': RUN_NAME,
    'pdb_s1': INITIAL_PDB,
    'pdb_s2': TARGET_PDB,
    'chain_s1': CHAIN,
    'chain_s2': CHAIN,
    'label_s1': INITIAL_PDB,
    'label_s2': TARGET_PDB,
    'source': 'user_input',
}

run_save_dir = os.path.join(BASE_SAVE_DIR, RUN_NAME)
os.makedirs(run_save_dir, exist_ok=True)

# Resume kontrolu
_pkl_path = os.path.join(run_save_dir, 'results.pkl')
all_results = []
_completed_keys = set()
if os.path.exists(_pkl_path):
    with open(_pkl_path, 'rb') as f:
        _loaded = pickle.load(f)
    for entry in _loaded:
        res = {k: v for k, v in entry.items() if k not in ('protein', 'direction', 'mode')}
        all_results.append((entry['protein'], entry['direction'], entry['mode'], res))
        _completed_keys.add(entry['direction'])
    print(f'RESUME: {len(all_results)} onceki run yuklendi')
    for n, d, dt, _ in all_results:
        print(f'  [SKIP] {d} ({dt})')

# UniProt sekans
uni_seq = None
if UNIPROT_ID:
    try:
        uni_seq = fetch_uniprot_sequence(UNIPROT_ID)
    except Exception as e:
        print(f'WARNING: UniProt fetch failed ({e})')

# ── [A] Initial -> Target ──
_key_a = f'{INITIAL_PDB}->{TARGET_PDB}'
if _key_a in _completed_keys:
    print(f'\n[A] {_key_a}  ** SKIP **')
else:
    print(f'\n[A] {INITIAL_PDB} -> {TARGET_PDB}')
    try:
        res_a = run_single_direction(converter, of3_cache, pair, direction='s1_to_s2',
                                     mode='directed', save_dir_override=run_save_dir)
        if res_a is not None:
            all_results.append((RUN_NAME, _key_a, 'open->closed', res_a))
    except Exception as e:
        print(f'  HATA: {e}')
        import traceback; traceback.print_exc()

# ── [B] Target -> Initial ──
_key_b = f'{TARGET_PDB}->{INITIAL_PDB}'
if _key_b in _completed_keys:
    print(f'\n[B] {_key_b}  ** SKIP **')
else:
    print(f'\n[B] {TARGET_PDB} -> {INITIAL_PDB}')
    try:
        res_b = run_single_direction(converter, of3_cache, pair, direction='s2_to_s1',
                                     mode='directed', save_dir_override=run_save_dir)
        if res_b is not None:
            all_results.append((RUN_NAME, _key_b, 'closed->open', res_b))
    except Exception as e:
        print(f'  HATA: {e}')
        import traceback; traceback.print_exc()

# ── [C] UniProt (opsiyonel) ──
if uni_seq:
    _key_c = f'{UNIPROT_ID}(uni)'
    if _key_c in _completed_keys:
        print(f'\n[C] {_key_c}  ** SKIP **')
    else:
        print(f'\n[C] UniProt {UNIPROT_ID}')
        try:
            res_c = run_single_direction(converter, of3_cache, pair, direction='s1_to_s2',
                                         mode='uniprot', uniprot_seq=uni_seq,
                                         save_dir_override=run_save_dir)
            if res_c is not None:
                all_results.append((RUN_NAME, _key_c, 'uniprot', res_c))
        except Exception as e:
            print(f'  HATA: {e}')
            import traceback; traceback.print_exc()

# ── Kaydet ──
save_data = []
for n, d, dt, res in all_results:
    entry = {k: v for k, v in res.items()}
    entry['protein'], entry['direction'], entry['mode'] = n, d, dt
    if 'trajectory_coords' in entry:
        entry['trajectory_coords'] = [
            c.cpu().numpy() if hasattr(c, 'cpu') else c for c in entry['trajectory_coords']
        ]
    save_data.append(entry)
with open(_pkl_path, 'wb') as f:
    pickle.dump(save_data, f)

print(f'\n{"="*60}')
print(f'TAMAMLANDI: {len(all_results)} run')
print(f'Saved: {_pkl_path}')
print(f'{"="*60}')


## 7. Sonuc Tablosu

In [ ]:
# ════════════════════════════════════════════════════════════════
#  SONUC TABLOSU
# ════════════════════════════════════════════════════════════════

if all_results:
    rows = []
    for name, direction, dir_type, res in all_results:
        rows.append({
            'Direction': direction,
            'Mode': dir_type,
            'Init TM': f'{res["init_tm"]:.4f}',
            'Best TM': f'{res["best_tm"]:.4f}',
            'Delta TM': f'{res["delta_tm"]:+.4f}',
            'Init RMSD': f'{res["init_rmsd"]:.2f}',
            'Final RMSD': f'{res["final_rmsd"]:.2f}',
            'Steps': res['n_steps_run'],
            'Accepted': res['n_accepted'],
            'Rollbacks': res['rollbacks'],
            'Time (s)': f'{res["wall_time"]:.0f}',
        })
    df = pd.DataFrame(rows)
    print(df.to_string(index=False))


## 8. Figure: TM Trajectory

In [ ]:
# ════════════════════════════════════════════════════════════════
#  TM TRAJECTORY: Her run icin step-by-step TM to target
# ════════════════════════════════════════════════════════════════

color_map = {'open->closed': '#e74c3c', 'closed->open': '#3498db', 'uniprot': '#27ae60'}

if all_results:
    n_runs = len(all_results)
    fig, axes = plt.subplots(1, n_runs, figsize=(6*n_runs, 5), squeeze=False)
    axes = axes[0]

    for idx, (name, direction, dir_type, res) in enumerate(all_results):
        ax = axes[idx]
        tms = res['step_tms_target']
        color = color_map.get(dir_type, '#7f8c8d')
        ax.plot(range(len(tms)), tms, 'o-', color=color, markersize=4, linewidth=1.5)
        ax.axhline(y=res['init_tm'], color='gray', linestyle='--', alpha=0.4, label='init')
        ax.set_xlabel('Step', fontsize=10)
        ax.set_ylabel('TM to Target', fontsize=10)
        ax.set_title(f'{direction}\n({dir_type})', fontsize=10)
        ax.set_ylim(max(0, min(tms)-0.05), min(1.0, max(tms)+0.05))
        ax.grid(True, alpha=0.2)
        ax.text(0.95, 0.05, f'Δ={res["delta_tm"]:+.3f}',
               transform=ax.transAxes, fontsize=9, ha='right', va='bottom',
               bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

    plt.suptitle(f'{RUN_NAME} — TM Trajectory', fontsize=13, fontweight='bold')
    plt.tight_layout(rect=[0, 0, 1, 0.93])
    p = os.path.join(run_save_dir, 'fig_tm_trajectory.png')
    fig.savefig(p, dpi=150, bbox_inches='tight'); plt.show()
    print(f'Saved: {p}')


## 9. Figure: Conformational Landscape

In [ ]:
# ════════════════════════════════════════════════════════════════
#  CONFORMATIONAL LANDSCAPE: TM/RMSD to Open vs Closed per step
#  + Experimental PDB yapilari (RCSB'den cekilir)
# ════════════════════════════════════════════════════════════════

import matplotlib.cm as cm
from matplotlib.lines import Line2D

# ── Experimental yapilari cek ──
exp_structures = []
if UNIPROT_ID:
    print(f'Fetching experimental structures for {UNIPROT_ID}...')
    try:
        pdb_ids = fetch_pdb_ids_for_uniprot(UNIPROT_ID, max_results=30)
        print(f'  Found {len(pdb_ids)} PDB entries')

        _chain = CHAIN.split(',')[0]
        open_ca, open_seq = fetch_ca(INITIAL_PDB, _chain)
        closed_ca, closed_seq = fetch_ca(TARGET_PDB, _chain)
        known_pdbs = {INITIAL_PDB.upper(), TARGET_PDB.upper()}

        for pid in pdb_ids:
            try:
                exp_ca, exp_seq = fetch_ca(pid, _chain)
            except Exception:
                try:
                    exp_ca, exp_seq = fetch_ca(pid, 'A')
                except Exception:
                    continue
            try:
                if exp_ca.shape[0] != open_ca.shape[0]:
                    ca1, ca2, _, _, _ = align_and_trim_ca(exp_ca, exp_seq, open_ca, open_seq)
                else:
                    ca1, ca2 = exp_ca, open_ca
                tm_open = tm_score(ca1, ca2)
                rmsd_open = compute_rmsd(ca1, ca2)
            except Exception:
                continue
            try:
                if exp_ca.shape[0] != closed_ca.shape[0]:
                    ca1, ca2, _, _, _ = align_and_trim_ca(exp_ca, exp_seq, closed_ca, closed_seq)
                else:
                    ca1, ca2 = exp_ca, closed_ca
                tm_closed = tm_score(ca1, ca2)
                rmsd_closed = compute_rmsd(ca1, ca2)
            except Exception:
                continue
            exp_structures.append({
                'pdb_id': pid.upper(),
                'tm_to_open': tm_open, 'tm_to_closed': tm_closed,
                'rmsd_to_open': rmsd_open, 'rmsd_to_closed': rmsd_closed,
                'is_reference': pid.upper() in known_pdbs,
            })
        print(f'  {len(exp_structures)} structures with valid metrics')
    except Exception as e:
        print(f'  WARNING: experimental fetch failed ({e})')

# ── Plot ──
if all_results:
    fig, axes = plt.subplots(1, 2, figsize=(18, 8))

    for panel_idx, metric_type in enumerate(['tm', 'rmsd']):
        ax = axes[panel_idx]

        # Experimental yildizlar
        for entry in exp_structures:
            if metric_type == 'tm':
                xv, yv = entry['tm_to_open'], entry['tm_to_closed']
            else:
                xv, yv = entry.get('rmsd_to_open'), entry.get('rmsd_to_closed')
                if xv is None or yv is None:
                    continue
            if entry['is_reference']:
                ax.scatter(xv, yv, marker='*', s=350, c='#e74c3c',
                          edgecolors='black', linewidths=0.8, zorder=20)
                ax.annotate(entry['pdb_id'], (xv, yv), textcoords='offset points',
                           xytext=(7, 7), fontsize=8, fontweight='bold',
                           bbox=dict(boxstyle='round,pad=0.2', fc='white', ec='gray', alpha=0.7))
            else:
                ax.scatter(xv, yv, marker='*', s=200, c='#2ecc71',
                          edgecolors='black', linewidths=0.5, zorder=15, alpha=0.8)
                ax.annotate(entry['pdb_id'], (xv, yv), textcoords='offset points',
                           xytext=(5, 5), fontsize=6, alpha=0.6)

        # Pipeline trajectory
        for name, direction, dir_type, res in all_results:
            if dir_type not in ('open->closed', 'closed->open'):
                continue
            if metric_type == 'tm':
                vals_tgt = res['step_tms_target']
                vals_src = res.get('step_tms_source', [])
            else:
                vals_tgt = res['step_rmsds_target']
                vals_src = res.get('step_rmsds_source', [])
            if not vals_src or len(vals_src) < 2:
                continue
            n_pts = min(len(vals_tgt), len(vals_src))
            if dir_type == 'open->closed':
                x_vals, y_vals = vals_src[:n_pts], vals_tgt[:n_pts]
            else:
                x_vals, y_vals = vals_tgt[:n_pts], vals_src[:n_pts]

            # pLDDT
            step_plddts = []
            acc_idx = 0
            for si in range(n_pts):
                if si == 0:
                    step_plddts.append(90.0); continue
                metrics = res['step_metrics']
                if acc_idx < len(metrics):
                    plddt = metrics[acc_idx].get('plddt_mean', None)
                    step_plddts.append(plddt if plddt is not None else 75.0)
                    acc_idx += 1
                else:
                    step_plddts.append(75.0)

            line_color = '#e74c3c' if dir_type == 'open->closed' else '#3498db'
            for si in range(n_pts - 1):
                alpha = 0.15 + 0.35 * (si / max(n_pts - 1, 1))
                ax.annotate('', xy=(x_vals[si+1], y_vals[si+1]),
                           xytext=(x_vals[si], y_vals[si]),
                           arrowprops=dict(arrowstyle='->', color=line_color,
                                         alpha=alpha, lw=1.2,
                                         connectionstyle='arc3,rad=0.05'))
            if n_pts > 2:
                ax.scatter(x_vals[1:-1], y_vals[1:-1], c=step_plddts[1:-1],
                          cmap='RdYlGn', norm=plt.Normalize(60, 90),
                          s=50, edgecolors='black', linewidths=0.3, zorder=8, alpha=0.7)
            norm = plt.Normalize(60, 90)
            cmap_fn = cm.get_cmap('RdYlGn')
            ax.scatter(x_vals[0], y_vals[0], c=[cmap_fn(norm(step_plddts[0]))],
                      s=180, edgecolors='black', linewidths=1.5, zorder=12, marker='o')
            ax.annotate('init', (x_vals[0], y_vals[0]), textcoords='offset points',
                       xytext=(6, -12), fontsize=7, fontweight='bold', color=line_color,
                       bbox=dict(boxstyle='round,pad=0.15', fc='white', ec=line_color, alpha=0.6))
            ax.scatter(x_vals[-1], y_vals[-1], c=[cmap_fn(norm(step_plddts[-1]))],
                      s=180, edgecolors='black', linewidths=1.5, zorder=12, marker='D')
            ax.annotate(f'S{n_pts-1}', (x_vals[-1], y_vals[-1]), textcoords='offset points',
                       xytext=(6, -12), fontsize=7, fontweight='bold', color=line_color,
                       bbox=dict(boxstyle='round,pad=0.15', fc='white', ec=line_color, alpha=0.6))
            if metric_type == 'tm':
                best_idx = int(np.argmax(vals_tgt[:n_pts]))
            else:
                best_idx = int(np.argmin(vals_tgt[:n_pts]))
            if best_idx not in (0, n_pts - 1):
                ax.scatter(x_vals[best_idx], y_vals[best_idx],
                          c=[cmap_fn(norm(step_plddts[best_idx]))],
                          s=180, edgecolors='gold', linewidths=2, zorder=13, marker='o')
                ax.annotate(f'best', (x_vals[best_idx], y_vals[best_idx]),
                           textcoords='offset points', xytext=(6, 6), fontsize=6,
                           fontweight='bold', color='#d4a017',
                           bbox=dict(boxstyle='round,pad=0.15', fc='white', ec='gold', alpha=0.6))

        if metric_type == 'tm':
            ax.set_xlabel('TM to Open', fontsize=12)
            ax.set_ylabel('TM to Closed', fontsize=12)
            ax.set_xlim(0.3, 1.05); ax.set_ylim(0.3, 1.05)
            ax.set_aspect('equal')
            ax.plot([0, 1], [0, 1], 'k--', alpha=0.12)
        else:
            ax.set_xlabel('RMSD to Open (A)', fontsize=12)
            ax.set_ylabel('RMSD to Closed (A)', fontsize=12)
            ax.set_aspect('equal')
            ax.plot([0, 30], [0, 30], 'k--', alpha=0.12)
        ax.grid(True, alpha=0.15, linestyle=':')
        ax.set_facecolor('#fafafa')

    sm = cm.ScalarMappable(norm=plt.Normalize(60, 90), cmap='RdYlGn')
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=axes[-1], shrink=0.5, pad=0.08, aspect=30)
    cbar.set_label('pLDDT', fontsize=10)

    legend_el = [
        Line2D([0], [0], marker='*', color='w', markerfacecolor='#e74c3c',
               markeredgecolor='black', markersize=14, label='Reference PDB'),
        Line2D([0], [0], marker='*', color='w', markerfacecolor='#2ecc71',
               markeredgecolor='black', markersize=12, label='Other Experimental'),
        Line2D([0], [0], marker='o', color='w', markerfacecolor='#888',
               markeredgecolor='black', markersize=8, label='Pipeline Step'),
        Line2D([0], [0], marker='D', color='w', markerfacecolor='#888',
               markeredgecolor='black', markersize=8, label='Final Step'),
        Line2D([0], [0], color='#e74c3c', linewidth=2, alpha=0.6, label=f'{INITIAL_PDB}\u2192{TARGET_PDB}'),
        Line2D([0], [0], color='#3498db', linewidth=2, alpha=0.6, label=f'{TARGET_PDB}\u2192{INITIAL_PDB}'),
    ]
    axes[0].legend(handles=legend_el, loc='lower left', fontsize=7, framealpha=0.8, edgecolor='gray')

    fig.suptitle(f'{RUN_NAME} \u2014 Conformational Landscape\n'
                 f'Stars=Experimental, Circles=Mode-Drive Trajectory (color=pLDDT)',
                 fontsize=13, fontweight='bold')
    plt.tight_layout(rect=[0, 0, 0.95, 0.93])
    p = os.path.join(run_save_dir, 'fig_landscape.png')
    fig.savefig(p, dpi=150, bbox_inches='tight'); plt.show()
    print(f'Saved: {p}')
